## Mart_Table (mart_customer_behaviour) GOLD LAYER INSERTION

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, count, countDistinct, avg, max as spark_max, min as spark_min,
    round as spark_round, col, when, datediff, ntile, date_trunc, current_date
)
from pyspark.sql import Window

#####What this does:

- Groups by customer (not order)
- Calculates Recency (days since last purchase), Frequency (order count), Monetary (total spend)
- Uses NTILE(5) to score each R/F/M on a 1-5 scale
- Segments customers: Champion (all 5s), Loyal, At Risk, Lost, etc.

In [0]:


# Load tables
df_fact = spark.table("olist_ecommerce_project.gold.fact_orders")
df_customers = spark.table("olist_ecommerce_project.gold.dim_customers")

# Note: fact_orders has customer_id (order-level), but we need customer_unique_id for grouping
# We'll add customer_unique_id to fact first by joining with slv_customers
df_slv_customers = spark.table("olist_ecommerce_project.silver.slv_customers")

# Join fact with Silver customers to get customer_unique_id
df_fact_with_unique = (
    df_fact
    .join(
        df_slv_customers.select("customer_id", "customer_unique_id", "customer_state"),
        on="customer_id",
        how="inner"
    )
)

# Step 1: Aggregate customer-level metrics by customer_unique_id
df_customer_agg = (
    df_fact_with_unique
    .groupBy("customer_unique_id", "customer_state")
    .agg(
        count("order_id").alias("total_orders"),
        spark_sum("total_order_value").alias("total_revenue"),
        spark_round(avg("total_order_value"), 2).alias("avg_order_value"),
        spark_min("order_purchase_timestamp").alias("first_purchase_date"),
        spark_max("order_purchase_timestamp").alias("last_purchase_date"),
        count(when(col("review_score").isNotNull(), 1)).alias("total_reviews")
    )
)

# Step 2: Add cohort month and recency
df_customer_agg = (
    df_customer_agg
    .withColumn(
        "cohort_month",
        date_trunc("month", col("first_purchase_date"))
    )
    .withColumn(
        "days_since_last_order",
        datediff(current_date(), col("last_purchase_date"))
    )
    .withColumn(
        "is_repeat_customer",
        when(col("total_orders") > 1, True).otherwise(False)
    )
)

# Step 3: Calculate RFM scores (1-5 scale using NTILE)
window_recency = Window.orderBy(col("days_since_last_order").desc())
window_frequency = Window.orderBy(col("total_orders"))
window_monetary = Window.orderBy(col("total_revenue"))

df_customer_rfm = (
    df_customer_agg
    .withColumn(
        "rfm_recency_score",
        ntile(5).over(window_recency)
    )
    .withColumn(
        "rfm_frequency_score",
        ntile(5).over(window_frequency)
    )
    .withColumn(
        "rfm_monetary_score",
        ntile(5).over(window_monetary)
    )
)

# Step 4: Calculate total RFM score
df_customer_rfm = (
    df_customer_rfm
    .withColumn(
        "rfm_total_score",
        col("rfm_recency_score") + col("rfm_frequency_score") + col("rfm_monetary_score")
    )
)

# Step 5: Segment customers based on RFM
df_customer_rfm = (
    df_customer_rfm
    .withColumn(
        "rfm_segment",
        when((col("rfm_recency_score") == 5) & (col("rfm_frequency_score") == 5) & (col("rfm_monetary_score") == 5), "Champion")
        .when((col("rfm_recency_score") >= 4) & (col("rfm_frequency_score") >= 4), "Loyal")
        .when((col("rfm_recency_score") >= 3) & (col("rfm_frequency_score") >= 3) & (col("rfm_monetary_score") >= 3), "Potential Loyal")
        .when((col("rfm_recency_score") <= 2) & (col("rfm_monetary_score") >= 4), "Cannot Lose")
        .when((col("rfm_recency_score") <= 2), "Lost")
        .otherwise("At Risk")
    )
)

# Select final columns
df_mart_customer = (
    df_customer_rfm.select(
        "customer_unique_id",
        "customer_state",
        "total_orders",
        "is_repeat_customer",
        "total_revenue",
        "avg_order_value",
        "total_reviews",
        "first_purchase_date",
        "last_purchase_date",
        "cohort_month",
        "days_since_last_order",
        "rfm_recency_score",
        "rfm_frequency_score",
        "rfm_monetary_score",
        "rfm_total_score",
        "rfm_segment"
    )
)

print("mart_customer_behavior rows:", df_mart_customer.count())
df_mart_customer.select(
    "customer_unique_id",
    "total_orders",
    "total_revenue",
    "rfm_segment",
    "days_since_last_order"
).show(10, truncate=False)



In [0]:
# Write to Gold
(
    df_mart_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.mart_customer_behavior")
)

print("mart_customer_behavior written successfully")